In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import scipy

import plotly
from plotly.graph_objects import Scatter
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
init_notebook_mode(connected=False)
import plotly.express as px

import cosmosdr

import cosmosdr.signal_acquisition as s_acq
import cosmosdr.signal_processing as s_proc
from cosmosdr.plotting import create_base_figure

try:
    sdr.close()
except:
    pass

In [ ]:
ADSB_FREQUENCY = 1090e6
# Total number of bits in a single ADSB message
ADSB_BITS = 112
# One microsecond timeslot for one bit, 'on' if signal is within the first half of this slot
ADSB_SLOT_LENGTH = 1 / 1e6

center_freq = ADSB_FREQUENCY
# reccomended upper limit of sample rate. Fast enough to oversample
sample_rate = 2e6
n_reads = 32
n_samples = 4096

# Code to learn

In [ ]:
import numpy as np
from scipy.signal import resample_poly, correlate
from fractions import Fraction

def iq_to_envelope(iq):
    """assumes `iq` is a 1D complex64 numpy array from your SDR, and sr is sample_rate (Hz)"""

    return np.abs(iq)

def resample_to_target(envelope, orig_sr, target_sr):
    # find integer up/down using Fraction
    ratio = Fraction(int(target_sr), int(orig_sr)).limit_denominator()
    up, down = ratio.numerator, ratio.denominator
    return resample_poly(envelope, up, down), target_sr

# # ADS-B preamble energy template (1 us slots, pulses at 0,0.5,1,3.5 us)
# def build_preamble_template(samples_per_us):
#     # create an envelope template at the target sample rate
#     sp = samples_per_us
#     length_us = 8  # preamble nominal 8 us length
#     L = length_us * sp
#     t = np.zeros(L, dtype=float)
#     # place narrow rectangles (0.5us pulses) at expected offsets (in us)
#     pulse_positions_us = [0.0, 0.5, 1.0, 3.5]
#     pulse_width_us = 0.5
#     w = int(round(pulse_width_us * sp))
#     for pos in pulse_positions_us:
#         start = int(round(pos * sp))
#         t[start:start+w] += 1.0
#     return t


# env = iq_to_envelope(iq)   # iq from your streamer
# env_up, new_sr = resample_to_target(env, orig_sr, target_sr)
# preambles, corr_norm = find_preambles(env_up, samples_per_us, threshold=0.5)

# for p in preambles:
#     bits = extract_bits_from_message(env_up, p, samples_per_us, message_us=112)
#     print("bits:", ''.join(map(str, bits[:56])))  # show first half for debug

### Sample at target SR, and upsample to higher rate

In [ ]:
# Example usage
orig_sr = 2.4e6
# choose integer samples per microsecond: 12 samples/us
# This is a good choice, because 6x2 = 12, and 5*2.4=12, so we can sample to 12, then subsample back to 1 block per 0.5us
target_sr = 12e6
samples_per_us = int(round(target_sr/1e6))  # 12

# Start up the SDR connection
try:
    sdr.close()
except:
    pass

sdr = s_acq.get_sdr(center_freq=center_freq, sample_rate=orig_sr)

s = s_acq.acquire_signal(sdr, n_reads=n_reads, n_samples=n_samples)

In [ ]:
s_df = pd.DataFrame(np.abs(s)).T

highest_peak_read = s_df.max().idxmax()

# Grab the column with the highest individual peak
signal_col = s_df[highest_peak_read]
print("highest index:", highest_peak_read)

iq = s[highest_peak_read]

In [ ]:

fig = px.bar(pd.DataFrame(iq).abs())

fig.update_traces(marker_line_width = 0,
                  selector=dict(type="bar"))

In [ ]:
# Set to abs, magnitude
env = iq_to_envelope(iq)

In [ ]:
env_up, new_sr = resample_to_target(env, orig_sr, target_sr)

In [ ]:
idx = env_up.argmax()
n_either_side = 750

In [ ]:
fig = px.bar(pd.DataFrame(env_up[idx-n_either_side:idx+n_either_side]).abs())

fig.update_traces(marker_line_width = 0,
                  selector=dict(type="bar"))

### Find the optimal phase starting position
- We don't know the exact timing of the pulses
- It could be anywhere from n, ..., n+11
- We can define the most optimal position as that which has the highest difference between neighboring 6-width blocks

In [ ]:
data = env_up[idx-n_either_side:idx+n_either_side]

scores = {}

In [ ]:
np.arange(len(data_phase)) // 1

In [ ]:
data_phase

In [ ]:
block_averages

In [ ]:
block_averages.shift(1)

In [ ]:
for phase in range(1, 20):
    data_phase = data[phase:]
    
    # 0,0,0,0,1,1,1,1...
    indices = np.arange(len(data_phase)) // 6
    
    data_phase = pd.Series(data_phase, index=indices)
    
    block_averages = data_phase.groupby(data_phase.index).mean()
    
    # calculate the differences between the blocks
    deltas = (block_averages - block_averages.shift(1)).abs()
    scores[phase] = deltas.mean()

In [ ]:
# We see the optimal split oscillates every 6 steps, as it should
px.line(pd.DataFrame(scores.values()))

In [ ]:
data.shape[0] % 6

In [ ]:
data[:-0]

In [ ]:
data[:-(data.shape[0] % phase)]

In [ ]:
data_phase

In [ ]:
scores

In [ ]:
data

In [ ]:
np.array(range(len(data)))

In [ ]:
(np.array(range(len(data))) / phase)

In [ ]:
data.index

# Interpreting aircraft signals

In [ ]:
1 / sample_rate

In [ ]:
ADSB_SLOT_LENGTH / 2

In [ ]:
# Assert that we are sampling at the same rate as the signal comms (so we can plot easily later
assert (1 / sample_rate) == ((ADSB_SLOT_LENGTH / 2))

In [ ]:
# Start up the SDR connection
try:
    sdr.close()
except:
    pass

sdr = s_acq.get_sdr(center_freq=center_freq, sample_rate=sample_rate)

s = s_acq.acquire_signal(sdr, n_reads=n_reads, n_samples=n_samples)


In [ ]:
s_df = pd.DataFrame(np.abs(s)).T

# Plot all the reads, see where there were peaks

In [ ]:
px.line(s_df.rolling(16).max()[::64])

### Plot the samples of the best candidate read
Given that this capture had the highest peak magnitude within it, it likely included an ASDB pulse

In [ ]:
# Grab the column with the highest individual peak
signal_col = s_df[s_df.max().idxmax()]

In [ ]:
fig = px.bar(signal_col)

fig.update_traces(marker_line_width = 0,
                  selector=dict(type="bar"))

# fig.update_layout(bargap=0,
#                   bargroupgap = 0,
#                  )


In [ ]:
low_cut = 0.15
signal_col.loc[signal_col < low_cut] = 0.1


In [ ]:
# mode S preamble, 8us
preamble = [1,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0]
assert(len(preamble)==16)
    

In [ ]:
fig = px.bar(signal_col)

fig.update_traces(marker_line_width = 0,
                  selector=dict(type="bar"))

# fig.update_layout(bargap=0,
#                   bargroupgap = 0,
#                  )


# Plot the on/off signalling

- If the peak is one colour, then the signal was in the first half of a slot
- If the peak is the other,  then the signal was in the second half of a slot

On or off depends on the start point, can't be predicted assessed ahead of time

In [ ]:
evens_indexer = signal_col.index % 2 == 0

In [ ]:
# Split the first half/second half into separate columns so they can be plotted differently
even = signal_col.reindex(signal_col.index[evens_indexer])
odd  = signal_col.reindex(signal_col.index[~evens_indexer])
even.name="even"
odd.name="odd"

# Plot the strongest signal to highlight 1s and 0s
- Within each 1us (microsecond, millionth of a second), there are two halves to the 'frame'
- If there is a signal peak within the first half, this is a 1
- If there is a signal peak within the second half, this is a 0

In [ ]:
# fig = px.bar(pd.concat([even, odd], axis=1))

# fig.update_traces(marker_line_width = 0,
#                   selector=dict(type="bar"))

In [ ]:
low_cut = 0.15